# Plant Leaves Super-Resolution v3 — cGAN + Global Residual

**Architecture**: Conditional GAN with RRDB Generator (23 blocks) and PatchGAN Discriminator.

**Key features:**
1. **Global Residual Learning** — `SR = Bicubic(LR) + Generator(LR)`. Generator predicts only the missing high-frequency detail.
2. **Two-Phase Training** — Phase 1 builds pixel-faithful foundation, Phase 2 adds GAN texture refinement.
3. **FFT Loss** — Frequency-domain penalty for recovering biological textures.
4. **Smart Ensemble** — Only checkpoints within 0.5 MAE of best are used for inference.
5. **ICNR PixelShuffle init** — Better upsampling starting point.
6. **EMA + 8-fold TTA** — Stable weights + test-time augmentation.

All weights randomly initialized. VGG19 used only for perceptual loss (provided weights).

In [1]:
import os
import sys
import csv
import time
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.nn.utils import clip_grad_norm_, spectral_norm
import torchvision.models as tv_models

warnings.filterwarnings('ignore')
csv.field_size_limit(sys.maxsize)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
Device   : cuda
GPU      : Tesla T4
VRAM     : 15.6 GB


In [2]:
class CFG:
    INPUT_DIR    = '/kaggle/input'
    TRAIN_HR_DIR = f'/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_High_Resolution'
    TRAIN_LR_DIR = f'/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution'
    TEST_LR_DIR  = f'/kaggle/input/competitions/plant-leaves-super-resolution-challenge/test_Low_Resolution'
    VGG_WEIGHTS  = f'/kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth'
    OUTPUT_DIR   = '/kaggle/working'

    # Architecture
    NUM_RRDB      = 23
    BASE_CHANNELS = 64
    GROWTH_CH     = 32

    # Phase 1: Pure regression (pixel-faithful foundation)
    PHASE1_EPOCHS = 250
    # Phase 2: Light GAN fine-tuning (texture refinement)
    PHASE2_EPOCHS = 100
    TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS   # 350

    BATCH_SIZE    = 16
    VAL_SPLIT     = 0.1
    SEED          = 42

    # Phase 1 LR
    G_LR          = 2e-4
    WARMUP_EPOCHS = 5

    # Phase 2 LR (much lower to preserve Phase 1 gains)
    G_LR_PHASE2   = 1e-5
    D_LR_PHASE2   = 5e-5

    # Loss weights
    W_CHARB  = 1.0
    W_PERCEP = 0.02
    W_FFT    = 0.05
    W_ADV    = 0.001     # very low adversarial weight (v2 used 0.005, which was too high)

    # EMA
    EMA_DECAY = 0.999

    # Gradient clipping
    GRAD_CLIP = 1.0

    # Checkpoints for ensemble
    ENSEMBLE_EPOCHS = [200, 225, 250, 275, 300, 325, 350]

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Config loaded.")
print(f"  RRDB blocks  : {CFG.NUM_RRDB}")
print(f"  Phase 1      : {CFG.PHASE1_EPOCHS} epochs (Charbonnier + Perceptual + FFT, NO GAN)")
print(f"  Phase 2      : {CFG.PHASE2_EPOCHS} epochs (+ light GAN, W_ADV={CFG.W_ADV})")
print(f"  EMA decay    : {CFG.EMA_DECAY}")
print(f"  Global residual: SR = Bicubic(LR) + Generator(LR)")

Config loaded.
  RRDB blocks  : 23
  Phase 1      : 250 epochs (Charbonnier + Perceptual + FFT, NO GAN)
  Phase 2      : 100 epochs (+ light GAN, W_ADV=0.001)
  EMA decay    : 0.999
  Global residual: SR = Bicubic(LR) + Generator(LR)


In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

## Dataset

In [4]:
class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, filenames, augment=True):
        self.lr_dir    = lr_dir
        self.hr_dir    = hr_dir
        self.filenames = filenames
        self.augment   = augment

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname  = self.filenames[idx]
        lr_img = np.array(Image.open(os.path.join(self.lr_dir, fname)).convert('RGB'), dtype=np.float32)
        hr_img = np.array(Image.open(os.path.join(self.hr_dir, fname)).convert('RGB'), dtype=np.float32)

        if self.augment:
            if random.random() > 0.5:
                lr_img = lr_img[:, ::-1, :].copy()
                hr_img = hr_img[:, ::-1, :].copy()
            if random.random() > 0.5:
                lr_img = lr_img[::-1, :, :].copy()
                hr_img = hr_img[::-1, :, :].copy()
            k = random.randint(0, 3)
            if k > 0:
                lr_img = np.rot90(lr_img, k, axes=(0, 1)).copy()
                hr_img = np.rot90(hr_img, k, axes=(0, 1)).copy()

        return (torch.from_numpy(lr_img / 255.0).permute(2, 0, 1).float(),
                torch.from_numpy(hr_img / 255.0).permute(2, 0, 1).float())


class TestDataset(Dataset):
    def __init__(self, lr_dir, filenames):
        self.lr_dir    = lr_dir
        self.filenames = filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        lr    = np.array(Image.open(os.path.join(self.lr_dir, fname)).convert('RGB'), dtype=np.float32)
        return torch.from_numpy(lr / 255.0).permute(2, 0, 1).float(), fname

## RRDB Generator with Global Residual Learning

The generator outputs a **residual** added to a bicubic upscale of the LR input.
The network only learns the missing high-frequency detail (edges, veins, lesions).

`SR = Bicubic(LR) + Generator(LR)`

In [5]:
def icnr_init(conv, scale=2):
    """ICNR initialization for PixelShuffle: initial output is nearest-neighbor."""
    ni, nf, k, _ = conv.weight.shape
    ni2 = int(ni / (scale ** 2))
    kernel = torch.zeros(ni2, nf, k, k)
    nn.init.kaiming_normal_(kernel, a=0.2, nonlinearity='leaky_relu')
    kernel = kernel.repeat_interleave(scale ** 2, dim=0)
    conv.weight.data.copy_(kernel)
    if conv.bias is not None:
        nn.init.zeros_(conv.bias)


class DenseLayer(nn.Module):
    def __init__(self, in_ch, gc=32):
        super().__init__()
        self.conv  = nn.Conv2d(in_ch, gc, 3, 1, 1, bias=True)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        nn.init.kaiming_normal_(self.conv.weight, a=0.2, nonlinearity='leaky_relu')
        nn.init.zeros_(self.conv.bias)

    def forward(self, x):
        return self.lrelu(self.conv(x))


class ResidualDenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.d1 = DenseLayer(nf, gc)
        self.d2 = DenseLayer(nf + gc, gc)
        self.d3 = DenseLayer(nf + gc * 2, gc)
        self.d4 = DenseLayer(nf + gc * 3, gc)
        self.d5 = nn.Conv2d(nf + gc * 4, nf, 3, 1, 1, bias=True)
        self.beta = 0.2
        nn.init.kaiming_normal_(self.d5.weight, a=0, nonlinearity='leaky_relu')
        nn.init.zeros_(self.d5.bias)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(torch.cat([x, d1], 1))
        d3 = self.d3(torch.cat([x, d1, d2], 1))
        d4 = self.d4(torch.cat([x, d1, d2, d3], 1))
        d5 = self.d5(torch.cat([x, d1, d2, d3, d4], 1))
        return x + d5 * self.beta


class RRDB(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(nf, gc)
        self.rdb2 = ResidualDenseBlock(nf, gc)
        self.rdb3 = ResidualDenseBlock(nf, gc)
        self.beta = 0.2

    def forward(self, x):
        return x + self.rdb3(self.rdb2(self.rdb1(x))) * self.beta


class RRDBGenerator(nn.Module):
    """RRDB Generator for 4x SR. Outputs a RESIDUAL — caller adds bicubic."""
    def __init__(self, in_ch=3, out_ch=3, nf=64, nb=16, gc=32):
        super().__init__()
        self.conv_first = nn.Conv2d(in_ch, nf, 3, 1, 1)
        self.trunk      = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.trunk_conv = nn.Conv2d(nf, nf, 3, 1, 1)

        self.up1_conv = nn.Conv2d(nf, nf * 4, 3, 1, 1)
        self.up1_ps   = nn.PixelShuffle(2)
        self.up2_conv = nn.Conv2d(nf, nf * 4, 3, 1, 1)
        self.up2_ps   = nn.PixelShuffle(2)

        self.hr_conv  = nn.Conv2d(nf, nf, 3, 1, 1)
        self.out_conv = nn.Conv2d(nf, out_ch, 3, 1, 1)
        self.lrelu    = nn.LeakyReLU(0.2, inplace=True)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2, nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        icnr_init(self.up1_conv, scale=2)
        icnr_init(self.up2_conv, scale=2)
        # Last conv outputs near-zero residual initially
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, x):
        f = self.conv_first(x)
        f = f + self.trunk_conv(self.trunk(f))
        f = self.lrelu(self.up1_ps(self.up1_conv(f)))
        f = self.lrelu(self.up2_ps(self.up2_conv(f)))
        return self.out_conv(self.lrelu(self.hr_conv(f)))


def bicubic_up(lr_tensor):
    """4x bicubic upscale on CPU (bypasses missing CUDA kernel on P100/T4)."""
    device = lr_tensor.device
    with torch.no_grad():
        up = F.interpolate(
            lr_tensor.cpu(), scale_factor=4,
            mode='bicubic', align_corners=False
        ).to(device)
    return up

## PatchGAN Discriminator (Conditional)

The discriminator evaluates 6-channel input: the SR (or HR) image
concatenated with the bicubic-upsampled LR. This makes it a
**conditional** GAN — the discriminator judges quality relative to
the input, not in absolute terms.

Spectral normalization is applied for training stability.

In [6]:
class PatchGANDiscriminator(nn.Module):
    """
    Conditional PatchGAN discriminator with spectral normalization.
    Input: 6 channels (SR/HR concatenated with bicubic-upsampled LR).
    Output: patch-level real/fake scores.
    """
    def __init__(self, in_ch=6):
        super().__init__()

        def disc_block(in_c, out_c, stride=2, norm=True):
            layers = [spectral_norm(nn.Conv2d(in_c, out_c, 4, stride, 1, bias=False))]
            if norm:
                layers.append(nn.InstanceNorm2d(out_c, affine=True))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.model = nn.Sequential(
            disc_block(in_ch,  64,  stride=2, norm=False),  # 128 -> 64
            disc_block(64,    128,  stride=2),               # 64  -> 32
            disc_block(128,   256,  stride=2),               # 32  -> 16
            disc_block(256,   512,  stride=1),               # 16  -> 15
            spectral_norm(nn.Conv2d(512, 1, 4, 1, 1))       # 15  -> 14
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2, nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, img, lr_up):
        return self.model(torch.cat([img, lr_up], dim=1))

## Losses, EMA, and Utilities

In [7]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps2 = eps ** 2

    def forward(self, x, y):
        return torch.sqrt((x - y) ** 2 + self.eps2).mean()


class FFTLoss(nn.Module):
    """Frequency-domain L1 loss on Fourier magnitude spectrum."""
    def forward(self, sr, hr):
        sr_fft = torch.fft.rfft2(sr, norm='ortho')
        hr_fft = torch.fft.rfft2(hr, norm='ortho')
        return F.l1_loss(sr_fft.abs(), hr_fft.abs())


class VGG19Features(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        vgg = tv_models.vgg19(pretrained=False)
        vgg.load_state_dict(torch.load(weights_path, map_location='cpu'))
        self.features = nn.Sequential(*list(vgg.features.children())[:36])
        for p in self.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        return self.features((x - self.mean) / self.std)


class PerceptualLoss(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        self.vgg = VGG19Features(weights_path)
        self.criterion = nn.L1Loss()

    def forward(self, sr, hr):
        return self.criterion(self.vgg(sr), self.vgg(hr))


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    @torch.no_grad()
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1 - self.decay)

    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup = {}


def compute_mae(sr, hr):
    return F.l1_loss(
        (sr.clamp(0, 1) * 255).round(),
        (hr.clamp(0, 1) * 255).round()
    ).item()


@torch.no_grad()
def validate(gen, loader, device):
    gen.eval()
    total_mae, count = 0.0, 0
    for lr, hr in loader:
        lr, hr = lr.to(device), hr.to(device)
        lr_up  = bicubic_up(lr)
        sr     = (gen(lr) + lr_up).clamp(0, 1)
        for b in range(sr.size(0)):
            total_mae += compute_mae(sr[b:b+1], hr[b:b+1])
            count += 1
    return total_mae / count

## Data Preparation

In [8]:
train_filenames = sorted(os.listdir(CFG.TRAIN_LR_DIR))
test_filenames  = sorted(os.listdir(CFG.TEST_LR_DIR))

indices = list(range(len(train_filenames)))
random.seed(CFG.SEED)
random.shuffle(indices)

val_size    = int(len(train_filenames) * CFG.VAL_SPLIT)
train_files = [train_filenames[i] for i in indices[val_size:]]
val_files   = [train_filenames[i] for i in indices[:val_size]]

train_dataset = SRDataset(CFG.TRAIN_LR_DIR, CFG.TRAIN_HR_DIR, train_files, augment=True)
val_dataset   = SRDataset(CFG.TRAIN_LR_DIR, CFG.TRAIN_HR_DIR, val_files,   augment=False)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train : {len(train_files)} samples ({len(train_loader)} batches)")
print(f"Val   : {len(val_files)} samples ({len(val_loader)} batches)")
print(f"Test  : {len(test_filenames)} images")

Train : 1478 samples (92 batches)
Val   : 164 samples (11 batches)
Test  : 495 images


## Model Instantiation

In [9]:
generator = RRDBGenerator(
    nf=CFG.BASE_CHANNELS, nb=CFG.NUM_RRDB, gc=CFG.GROWTH_CH
).to(CFG.DEVICE)

discriminator = PatchGANDiscriminator(in_ch=6).to(CFG.DEVICE)

print(f"Generator params     : {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator params : {sum(p.numel() for p in discriminator.parameters()):,}")

# Verify global residual: initial output should be near bicubic
with torch.no_grad():
    dummy_lr  = torch.randn(1, 3, 32, 32).to(CFG.DEVICE)
    dummy_res = generator(dummy_lr)
    print(f"Initial residual magnitude: {dummy_res.abs().mean():.6f} (should be near 0)")
    del dummy_lr, dummy_res

# Losses
charb_loss_fn  = CharbonnierLoss()
fft_loss_fn    = FFTLoss()
percep_loss_fn = PerceptualLoss(CFG.VGG_WEIGHTS).to(CFG.DEVICE)
percep_loss_fn.eval()
adv_loss_fn    = nn.BCEWithLogitsLoss()
print("Losses: Charbonnier + FFT + VGG19 Perceptual + BCEWithLogits (GAN)")

# EMA
ema = EMA(generator, decay=CFG.EMA_DECAY)
print(f"EMA initialized (decay={CFG.EMA_DECAY})")

Generator params     : 16,919,555
Discriminator params : 2,768,641
Initial residual magnitude: 0.000000 (should be near 0)
Losses: Charbonnier + FFT + VGG19 Perceptual + BCEWithLogits (GAN)
EMA initialized (decay=0.999)


## Bicubic Baseline (Score Floor)

In [10]:
bicubic_csv_path = os.path.join(CFG.OUTPUT_DIR, 'submission_bicubic.csv')
with open(bicubic_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Id', 'Pixels'])
    for fname in test_filenames:
        lr_img = Image.open(os.path.join(CFG.TEST_LR_DIR, fname)).convert('RGB')
        hr_img = lr_img.resize((128, 128), Image.BICUBIC)
        pixels = np.array(hr_img, dtype=np.uint8).flatten()
        writer.writerow([fname, ' '.join(map(str, pixels))])
print(f"Bicubic baseline saved: {bicubic_csv_path}")

Bicubic baseline saved: /kaggle/working/submission_bicubic.csv


## Phase 1 — Pixel-Faithful Regression (No GAN)

Build a rock-solid pixel-faithful generator using Charbonnier + FFT + Perceptual
losses before introducing the adversarial component. Global residual learning
means the generator starts at bicubic quality and gradually learns to add
the missing high-frequency biological detail.

In [11]:
best_val_mae       = float('inf')
best_epoch         = 0
best_ema_ckpt_path = os.path.join(CFG.OUTPUT_DIR, 'best_ema.pth')
saved_checkpoints  = []
history            = []

optimizer_G = optim.Adam(generator.parameters(), lr=CFG.G_LR, betas=(0.9, 0.999))
scheduler_G = CosineAnnealingLR(optimizer_G, T_max=CFG.PHASE1_EPOCHS, eta_min=1e-7)

print(f"Phase 1: {CFG.PHASE1_EPOCHS} epochs, lr={CFG.G_LR}, NO adversarial loss")
print(f"{'Epoch':>5}  {'train':>8}  {'val':>8}  {'ema_val':>8}  {'loss':>8}  {'lr':>10}  {'time':>5}")
print("-" * 65)

for epoch in range(1, CFG.PHASE1_EPOCHS + 1):
    t0 = time.time()
    generator.train()

    # Linear warmup
    if epoch <= CFG.WARMUP_EPOCHS:
        for pg in optimizer_G.param_groups:
            pg['lr'] = CFG.G_LR * (epoch / CFG.WARMUP_EPOCHS)

    sum_loss, sum_mae, n = 0.0, 0.0, 0

    for lr, hr in train_loader:
        lr = lr.to(CFG.DEVICE)
        hr = hr.to(CFG.DEVICE)
        lr_up = bicubic_up(lr)

        optimizer_G.zero_grad()
        sr = (generator(lr) + lr_up)

        loss_c = charb_loss_fn(sr, hr)
        loss_f = fft_loss_fn(sr, hr)
        loss_p = percep_loss_fn(sr.clamp(0, 1), hr)
        loss   = CFG.W_CHARB * loss_c + CFG.W_FFT * loss_f + CFG.W_PERCEP * loss_p

        loss.backward()
        clip_grad_norm_(generator.parameters(), CFG.GRAD_CLIP)
        optimizer_G.step()
        ema.update(generator)

        with torch.no_grad():
            sum_mae += compute_mae(sr, hr)
        sum_loss += loss.item()
        n += 1

    if epoch > CFG.WARMUP_EPOCHS:
        scheduler_G.step()

    train_mae = sum_mae / n
    avg_loss  = sum_loss / n

    val_mae = validate(generator, val_loader, CFG.DEVICE)

    ema.apply_shadow(generator)
    ema_val_mae = validate(generator, val_loader, CFG.DEVICE)

    is_best = ema_val_mae < best_val_mae
    if is_best:
        best_val_mae = ema_val_mae
        best_epoch   = epoch
        torch.save(generator.state_dict(), best_ema_ckpt_path)

    if epoch in CFG.ENSEMBLE_EPOCHS:
        ckpt_path = os.path.join(CFG.OUTPUT_DIR, f'ema_ep{epoch:03d}.pth')
        torch.save(generator.state_dict(), ckpt_path)
        saved_checkpoints.append(ckpt_path)

    ema.restore(generator)

    history.append({
        'epoch': epoch, 'train_mae': round(train_mae, 3),
        'val_mae': round(val_mae, 3), 'ema_val_mae': round(ema_val_mae, 3),
        'loss': round(avg_loss, 5), 'phase': 1,
    })

    elapsed = time.time() - t0
    lr_now  = optimizer_G.param_groups[0]['lr']
    tag     = ' <-- best' if is_best else ''

    print(f"{epoch:5d}  {train_mae:8.3f}  {val_mae:8.3f}  {ema_val_mae:8.3f}  "
          f"{avg_loss:8.4f}  {lr_now:10.2e}  {elapsed:4.0f}s{tag}")

print()
print(f"Phase 1 complete. Best EMA val MAE: {best_val_mae:.3f} (epoch {best_epoch})")

Phase 1: 250 epochs, lr=0.0002, NO adversarial loss
Epoch     train       val   ema_val      loss          lr   time
-----------------------------------------------------------------
    1    20.886    18.328    18.894    0.0926    4.00e-05    58s <-- best
    2    18.087    18.211    18.973    0.0768    8.00e-05    62s
    3    17.838    18.077    19.131    0.0758    1.20e-04    66s
    4    17.680    18.009    19.037    0.0751    1.60e-04    66s
    5    17.635    17.963    18.935    0.0749    2.00e-04    65s
    6    17.544    17.907    18.829    0.0745    2.00e-04    65s <-- best
    7    17.510    17.825    18.687    0.0744    2.00e-04    65s <-- best
    8    17.455    17.845    18.559    0.0741    2.00e-04    65s <-- best
    9    17.431    17.812    18.441    0.0740    2.00e-04    66s <-- best
   10    17.384    17.750    18.338    0.0738    2.00e-04    66s <-- best
   11    17.395    17.751    18.241    0.0738    2.00e-04    65s <-- best
   12    17.365    17.721    18.164    

## Phase 2 — cGAN Fine-Tuning (Light Adversarial)

Add a very small adversarial signal (W_ADV=0.001) from the PatchGAN
discriminator to sharpen biological textures. Phase 1 established a strong
pixel-faithful model; Phase 2 refines texture detail with minimal MAE risk.

Key safeguards:
- Very low adversarial weight (5x lower than v2)
- Low learning rates for both G and D
- Label smoothing on real labels (0.9)
- EMA continues to shield against adversarial noise
- Smart ensemble will exclude any checkpoint where MAE drifts

In [12]:
# Load best Phase 1 EMA checkpoint as starting point for Phase 2
generator.load_state_dict(torch.load(best_ema_ckpt_path, map_location=CFG.DEVICE))
print(f"Loaded Phase 1 best EMA checkpoint (epoch {best_epoch}, val MAE {best_val_mae:.3f})")

optimizer_G2 = optim.Adam(generator.parameters(), lr=CFG.G_LR_PHASE2, betas=(0.9, 0.999))
optimizer_D  = optim.Adam(discriminator.parameters(), lr=CFG.D_LR_PHASE2, betas=(0.9, 0.999))
scheduler_G2 = CosineAnnealingLR(optimizer_G2, T_max=CFG.PHASE2_EPOCHS, eta_min=1e-7)
scheduler_D  = CosineAnnealingLR(optimizer_D,  T_max=CFG.PHASE2_EPOCHS, eta_min=1e-7)

# Re-initialize EMA from loaded checkpoint
ema = EMA(generator, decay=CFG.EMA_DECAY)

print(f"Phase 2: {CFG.PHASE2_EPOCHS} epochs, G_lr={CFG.G_LR_PHASE2}, D_lr={CFG.D_LR_PHASE2}, W_ADV={CFG.W_ADV}")
print(f"{'Epoch':>5}  {'train':>8}  {'val':>8}  {'ema_val':>8}  {'g_loss':>8}  {'d_loss':>8}  {'time':>5}")
print("-" * 70)

for epoch in range(CFG.PHASE1_EPOCHS + 1, CFG.TOTAL_EPOCHS + 1):
    t0 = time.time()
    generator.train()
    discriminator.train()

    sum_g_loss, sum_d_loss, sum_mae, n = 0.0, 0.0, 0.0, 0

    for lr, hr in train_loader:
        lr = lr.to(CFG.DEVICE)
        hr = hr.to(CFG.DEVICE)
        lr_up = bicubic_up(lr)

        sr = (generator(lr) + lr_up).clamp(0, 1)

        # ---------- Update Discriminator ----------
        optimizer_D.zero_grad()
        d_real = discriminator(hr, lr_up)
        d_fake = discriminator(sr.detach(), lr_up)
        # Label smoothing: real=0.9 to weaken D slightly
        real_label = torch.full_like(d_real, 0.9)
        fake_label = torch.zeros_like(d_fake)
        loss_d = (adv_loss_fn(d_real, real_label) + adv_loss_fn(d_fake, fake_label)) * 0.5
        loss_d.backward()
        clip_grad_norm_(discriminator.parameters(), CFG.GRAD_CLIP)
        optimizer_D.step()

        # ---------- Update Generator ----------
        optimizer_G2.zero_grad()
        sr = (generator(lr) + lr_up).clamp(0, 1)
        d_fake_for_g = discriminator(sr, lr_up)

        loss_c = charb_loss_fn(sr, hr)
        loss_f = fft_loss_fn(sr, hr)
        loss_p = percep_loss_fn(sr, hr)
        loss_a = adv_loss_fn(d_fake_for_g, torch.ones_like(d_fake_for_g))
        loss_g = (CFG.W_CHARB * loss_c + CFG.W_FFT * loss_f
                  + CFG.W_PERCEP * loss_p + CFG.W_ADV * loss_a)

        loss_g.backward()
        clip_grad_norm_(generator.parameters(), CFG.GRAD_CLIP)
        optimizer_G2.step()
        ema.update(generator)

        with torch.no_grad():
            sum_mae += compute_mae(sr, hr)
        sum_g_loss += loss_g.item()
        sum_d_loss += loss_d.item()
        n += 1

    scheduler_G2.step()
    scheduler_D.step()

    train_mae  = sum_mae / n
    avg_g_loss = sum_g_loss / n
    avg_d_loss = sum_d_loss / n

    val_mae = validate(generator, val_loader, CFG.DEVICE)

    ema.apply_shadow(generator)
    ema_val_mae = validate(generator, val_loader, CFG.DEVICE)

    is_best = ema_val_mae < best_val_mae
    if is_best:
        best_val_mae = ema_val_mae
        best_epoch   = epoch
        torch.save(generator.state_dict(), best_ema_ckpt_path)

    if epoch in CFG.ENSEMBLE_EPOCHS:
        ckpt_path = os.path.join(CFG.OUTPUT_DIR, f'ema_ep{epoch:03d}.pth')
        torch.save(generator.state_dict(), ckpt_path)
        saved_checkpoints.append(ckpt_path)

    ema.restore(generator)

    history.append({
        'epoch': epoch, 'train_mae': round(train_mae, 3),
        'val_mae': round(val_mae, 3), 'ema_val_mae': round(ema_val_mae, 3),
        'g_loss': round(avg_g_loss, 5), 'd_loss': round(avg_d_loss, 4),
        'phase': 2,
    })

    elapsed = time.time() - t0
    tag     = ' <-- best' if is_best else ''

    print(f"{epoch:5d}  {train_mae:8.3f}  {val_mae:8.3f}  {ema_val_mae:8.3f}  "
          f"{avg_g_loss:8.4f}  {avg_d_loss:8.4f}  {elapsed:4.0f}s{tag}")

# Save history
pd.DataFrame(history).to_csv(os.path.join(CFG.OUTPUT_DIR, 'training_history.csv'), index=False)

print()
print(f"Training complete. Best EMA val MAE: {best_val_mae:.3f} (epoch {best_epoch})")
print(f"Ensemble checkpoints saved: {len(saved_checkpoints)}")

Loaded Phase 1 best EMA checkpoint (epoch 114, val MAE 17.352)
Phase 2: 100 epochs, G_lr=1e-05, D_lr=5e-05, W_ADV=0.001
Epoch     train       val   ema_val    g_loss    d_loss   time
----------------------------------------------------------------------
  251    16.735    17.400    17.353    0.0721    0.5303    90s
  252    16.770    17.476    17.356    0.0733    0.2960    89s
  253    16.852    17.572    17.362    0.0743    0.2571    89s
  254    16.925    17.714    17.371    0.0748    0.2494    89s
  255    17.028    17.802    17.381    0.0754    0.2558    90s
  256    17.161    17.946    17.393    0.0757    0.2640    89s
  257    17.249    18.019    17.406    0.0761    0.2554    89s
  258    17.337    18.094    17.420    0.0763    0.2585    89s
  259    17.365    18.165    17.435    0.0765    0.2544    90s
  260    17.439    18.163    17.452    0.0770    0.2490    89s
  261    17.499    18.174    17.471    0.0772    0.2487    89s
  262    17.540    18.271    17.490    0.0773    0.25

## Inference — Smart Ensemble + 8-Fold TTA

In [13]:
def tta_predict(model, lr_tensor, device):
    """8-fold TTA with global residual."""
    preds = []
    for flip in [False, True]:
        for k in range(4):
            x = lr_tensor.clone()
            if flip:
                x = torch.flip(x, dims=[-1])
            if k > 0:
                x = torch.rot90(x, k, dims=[-2, -1])
            with torch.no_grad():
                x_dev = x.to(device)
                x_up  = bicubic_up(x_dev)
                sr    = (model(x_dev) + x_up).clamp(0, 1)
            if k > 0:
                sr = torch.rot90(sr, -k, dims=[-2, -1])
            if flip:
                sr = torch.flip(sr, dims=[-1])
            preds.append(sr.cpu())
    return torch.stack(preds).mean(0)


# --- Smart Ensemble: only checkpoints within 0.5 MAE of best ---
all_candidates = sorted(set(saved_checkpoints + [best_ema_ckpt_path]))

ckpt_scores = []
print("Evaluating ensemble checkpoints:")
for p in all_candidates:
    generator.load_state_dict(torch.load(p, map_location=CFG.DEVICE))
    generator.eval()
    v = validate(generator, val_loader, CFG.DEVICE)
    ckpt_scores.append((p, v))
    print(f"  {os.path.basename(p):30s}  val_mae={v:.3f}")

# Filter: keep only good ones
best_score = min(v for _, v in ckpt_scores)
selected = [(p, v) for p, v in ckpt_scores if v <= best_score + 0.5]

print(f"\nSelected {len(selected)} / {len(all_candidates)} checkpoints "
      f"(within 0.5 of best {best_score:.3f}):")
for p, v in selected:
    print(f"  {os.path.basename(p):30s}  val_mae={v:.3f}")

selected_paths = [p for p, v in selected]

# Run ensemble TTA
test_dataset = TestDataset(CFG.TEST_LR_DIR, test_filenames)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

total_fwd = len(selected_paths) * 8
print(f"\nRunning ensemble TTA on {len(test_filenames)} images...")
print(f"  {len(selected_paths)} checkpoints x 8 TTA = {total_fwd} forward passes per image")

submission_rows = []
for i, (lr, fname) in enumerate(test_loader):
    fname = fname[0]
    ckpt_preds = []
    for ckpt_path in selected_paths:
        generator.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE))
        generator.eval()
        ckpt_preds.append(tta_predict(generator, lr, CFG.DEVICE))

    avg_sr = torch.stack(ckpt_preds).mean(0)
    sr_img = (avg_sr.squeeze(0).permute(1, 2, 0).clamp(0, 1) * 255).round().byte().numpy()
    submission_rows.append((fname, ' '.join(map(str, sr_img.flatten()))))

    if (i + 1) % 50 == 0 or (i + 1) == len(test_filenames):
        print(f"  {i+1}/{len(test_filenames)}")

print("Ensemble TTA inference complete.")

Evaluating ensemble checkpoints:
  best_ema.pth                    val_mae=17.352
  ema_ep200.pth                   val_mae=17.524
  ema_ep225.pth                   val_mae=17.574
  ema_ep250.pth                   val_mae=17.597
  ema_ep275.pth                   val_mae=17.846
  ema_ep300.pth                   val_mae=18.439
  ema_ep325.pth                   val_mae=18.693
  ema_ep350.pth                   val_mae=18.802

Selected 5 / 8 checkpoints (within 0.5 of best 17.352):
  best_ema.pth                    val_mae=17.352
  ema_ep200.pth                   val_mae=17.524
  ema_ep225.pth                   val_mae=17.574
  ema_ep250.pth                   val_mae=17.597
  ema_ep275.pth                   val_mae=17.846

Running ensemble TTA on 495 images...
  5 checkpoints x 8 TTA = 40 forward passes per image
  50/495
  100/495
  150/495
  200/495
  250/495
  300/495
  350/495
  400/495
  450/495
  495/495
Ensemble TTA inference complete.


## Write Submission CSV

In [14]:
submission_csv_path = os.path.join(CFG.OUTPUT_DIR, 'submission.csv')
with open(submission_csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Id', 'Pixels'])
    for fname, pixel_str in submission_rows:
        writer.writerow([fname, pixel_str])

# Verify
with open(submission_csv_path, 'r') as f:
    reader = csv.reader(f)
    next(reader)
    first_row = next(reader)
    n_pixels  = len(first_row[1].split(' '))

print(f"Submission written : {submission_csv_path}")
print(f"Rows               : {len(submission_rows)}")
print(f"Pixels per row     : {n_pixels}  (expected: 49152)")
assert n_pixels == 49152
print()
print("-" * 50)
print(f"Best EMA val MAE   : {best_val_mae:.3f}  (epoch {best_epoch})")
print(f"Ensemble size      : {len(selected_paths)} checkpoints x 8 TTA")
print(f"Submission         : {submission_csv_path}")
print(f"Bicubic fallback   : {bicubic_csv_path}")
print("-" * 50)

Submission written : /kaggle/working/submission.csv
Rows               : 495
Pixels per row     : 49152  (expected: 49152)

--------------------------------------------------
Best EMA val MAE   : 17.352  (epoch 114)
Ensemble size      : 5 checkpoints x 8 TTA
Submission         : /kaggle/working/submission.csv
Bicubic fallback   : /kaggle/working/submission_bicubic.csv
--------------------------------------------------
